# Task 4: Advanced Statistical Modeling, Forecasting, Segmentation, and Predictive Analytics

A reproducible end-to-end retail analytics pipeline. Execute cells top to bottom.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.generate_data import save_transactions
from src.statistical_analysis import descriptive_statistics, run_hypothesis_tests
from src.time_series import prepare_weekly_sales, adf_report, forecast_sales, save_decomposition
from src.customer_segmentation import build_rfm, fit_segmentation
from src.predictive_modeling import run_predictive_models
for folder in ['data/processed','figures/statistics','figures/time_series','figures/clustering','figures/predictive','models']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / 'data/processed/customer_transactions_task4.csv'
FIGURES = ROOT / 'figures'
MODELS = ROOT / 'models'


## Section 0: Reproducible Synthetic Data

Generates exactly 2,500 transaction records across 2024-2025 with customer, order, service, rating, and churn fields.

In [ ]:
data = save_transactions(DATA_PATH)
print(data.shape, data['Customer_ID'].nunique(), data['Order_Date'].min(), data['Order_Date'].max())
display(data.head())


## Section 1: Advanced Statistical Analysis

Descriptive moments, Welch two-sample t-test, Region-Churn chi-square, one-way ANOVA, and 95% confidence intervals quantify differences and uncertainty. Reject a null only when p < 0.05.

In [ ]:
continuous = ['Order_Value','Discount_Pct','Delivery_Days','Recency_Days','Frequency','Customer_Rating']
display(descriptive_statistics(data, continuous).round(3))
statistics = run_hypothesis_tests(data)
for test_name, result in statistics.items(): print(test_name, result)
fig, axis = plt.subplots(figsize=(9, 4)); data.boxplot(column='Order_Value', by='Category', ax=axis); fig.suptitle(''); axis.set_title('Order value by category'); fig.tight_layout(); fig.savefig(FIGURES / 'statistics/category_anova_boxplot.png', dpi=150); plt.show()


## Section 2: Time Series Analysis and Forecasting

Dates are indexed and resampled to Monday-ending weeks; missing periods are forward-filled. ADF is run on levels and first differences. The ARIMA holdout forecast reports MAE, RMSE, MAPE and 95% confidence bands.

In [ ]:
weekly = prepare_weekly_sales(data)
print('ADF level:', adf_report(weekly))
print('ADF differenced:', adf_report(weekly.diff().dropna()))
save_decomposition(weekly, FIGURES / 'time_series/seasonal_decomposition.png')
forecast = forecast_sales(weekly, horizon=12, figure_path=FIGURES / 'time_series/arima_forecast_12weeks.png')
print('Forecast metrics:', forecast['metrics']); display(forecast['forecast'].to_frame('Forecast').round(2))


## Section 3: Customer Segmentation

Customer-level RFM features are standardized, evaluated for k=2..8 using WCSS and silhouette scores, clustered with K-Means, and projected with PCA. Persona recommendations are printed from the profile table.

In [ ]:
rfm = build_rfm(data)
segmentation = fit_segmentation(rfm, FIGURES / 'clustering')
segmented = segmentation['rfm']; segmented.to_csv(ROOT / 'data/processed/rfm_segmented_customers.csv', index=False)
joblib.dump(segmentation['model'], MODELS / 'kmeans_rfm_model.pkl')
display(segmentation['profile']); print('Selected k:', segmentation['selected_k'])
for cluster, row in segmentation['profile'].iterrows(): print(f'Cluster {cluster}: recency={row.Recency:.1f}, frequency={row.Frequency:.1f}, monetary={row.Monetary:.1f}; tailor retention, win-back, or cross-sell actions.')


## Section 4: Predictive Modeling

Linear regression predicts Order_Value from Recency, Frequency, Discount_Pct, and Delivery_Days. Logistic regression and a constrained decision tree classify Churn. Confusion matrices, Accuracy, Precision, Recall, F1, ROC-AUC, coefficients, and limitations are included.

In [ ]:
predictive = run_predictive_models(data, FIGURES / 'predictive')
joblib.dump(predictive['regression']['model'], MODELS / 'linear_order_value_model.pkl')
joblib.dump(predictive['classification']['logistic']['model'], MODELS / 'logistic_churn_model.pkl')
joblib.dump(predictive['classification']['decision_tree']['model'], MODELS / 'decision_tree_churn.pkl')
print('Regression:', predictive['regression']['metrics'])
for name in ['logistic','decision_tree']:
    result = predictive['classification'][name]; print(name, {metric: result[metric] for metric in ['accuracy','precision','recall','f1','roc_auc']}); print(result['confusion_matrix'])


## Section 5: Deliverables and Documentation

README.md contains setup, layout, limitations, and a metric-filled LinkedIn completion post template. Model results are associations on synthetic data: validate temporal drift, leakage, calibration, class costs, and causal assumptions before deployment.

Completion post metrics can be copied from the cells above: forecast MAE/RMSE/MAPE, regression R², and best churn ROC-AUC.